# Assignment 2 (simple version): AI vs Real with CLIP
Baseline: ResNet50 features. SOTA: CLIP ViT-L/14 features. Same classifier (logistic regression) for both.
Steps: load data, extract features, compare, ablation, explainability, latency, export for the app.
Runtime: GPU (Runtime -> Change runtime type -> T4 GPU). The dataset must be on Google Drive: `MyDrive/ai_vs_real/<class folders>`.

In [ ]:
!pip -q install open_clip_torch

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%%writefile detector.py
# Shared helpers for the A2 project: image encoders, occlusion heatmap, nearest neighbours.
# Used by both the Colab notebook and the Gradio app (app.py).
import numpy as np
import torch
from PIL import Image, ImageOps, ImageDraw


def load_image(path):
    im = Image.open(path)
    im.load()
    return im.convert("RGB")


class _Encoder:
    """Base class: turns a list of PIL images into a feature matrix."""
    l2 = False

    def _init_device(self, device=None):
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.dtype = torch.float16 if self.device == "cuda" else torch.float32

    def move(self, device):
        self.device = device
        self.dtype = torch.float16 if device == "cuda" else torch.float32
        self.model.to(device=device, dtype=self.dtype)
        return self

    @torch.no_grad()
    def encode(self, imgs, bs=64):
        out = []
        for i in range(0, len(imgs), bs):
            x = torch.stack([self.preprocess(im) for im in imgs[i:i + bs]])
            x = x.to(self.device, self.dtype)
            f = self._forward(x).float()
            if self.l2:
                f = f / f.norm(dim=-1, keepdim=True)
            out.append(f.cpu().numpy())
        return np.concatenate(out)


class ClipEncoder(_Encoder):
    """Frozen CLIP image encoder (open_clip). L2-normalised embeddings."""
    l2 = True

    def __init__(self, arch="ViT-L-14", pretrained="openai", device=None):
        import open_clip
        self._init_device(device)
        model, _, preprocess = open_clip.create_model_and_transforms(arch, pretrained=pretrained)
        self.model = model.eval().to(device=self.device, dtype=self.dtype)
        self.preprocess = preprocess
        self.name = f"CLIP {arch} ({pretrained})"

    def _forward(self, x):
        return self.model.encode_image(x)


class ResNetEncoder(_Encoder):
    """ImageNet ResNet50 feature extractor (2048-d). Used as the A1-style baseline."""

    def __init__(self, device=None):
        import torchvision
        self._init_device(device)
        w = torchvision.models.ResNet50_Weights.IMAGENET1K_V2
        m = torchvision.models.resnet50(weights=w)
        m.fc = torch.nn.Identity()
        self.model = m.eval().to(device=self.device, dtype=self.dtype)
        self.preprocess = w.transforms()
        self.name = "ResNet50 (ImageNet)"

    def _forward(self, x):
        return self.model(x)


def occlusion_heatmap(encoder, clf, img, grid=7, target=None, fill=(124, 116, 104)):
    """Model-agnostic explanation: slide a grey square over the image and measure
    how much the probability of the target class drops. Big drop = important region.
    Returns (heat[grid, grid], base_image_224)."""
    base = ImageOps.fit(img.convert("RGB"), (224, 224), Image.BICUBIC)
    p0 = clf.predict_proba(encoder.encode([base]))[0]
    t = int(p0.argmax()) if target is None else int(target)
    step = 224 / grid
    variants = []
    for r in range(grid):
        for c in range(grid):
            v = base.copy()
            ImageDraw.Draw(v).rectangle(
                [int(c * step), int(r * step), int((c + 1) * step) - 1, int((r + 1) * step) - 1], fill=fill)
            variants.append(v)
    ps = clf.predict_proba(encoder.encode(variants))[:, t]
    heat = (p0[t] - ps).reshape(grid, grid)
    return heat, base


def heat_overlay(base, heat, strength=0.65):
    """Blend a coarse heatmap (positive = supports the decision) onto the image."""
    import matplotlib
    h = np.clip(heat, 0, None)
    h = h / (h.max() + 1e-8)
    h_img = Image.fromarray((h * 255).astype(np.uint8)).resize(base.size, Image.BICUBIC)
    a = np.asarray(h_img, dtype=np.float32)[..., None] / 255.0
    rgb = matplotlib.colormaps["jet"](a[..., 0])[..., :3] * 255.0
    out = np.asarray(base.convert("RGB"), dtype=np.float32) * (1 - a * strength) + rgb * (a * strength)
    return Image.fromarray(out.clip(0, 255).astype(np.uint8))


def nearest_neighbors(q, gallery, k=5):
    """Cosine-similarity neighbours (embeddings are L2-normalised)."""
    sims = gallery @ q
    idx = np.argsort(-sims)[:k]
    return idx, sims[idx]

In [ ]:
import os, time, shutil, joblib
import numpy as np, pandas as pd, matplotlib.pyplot as plt, torch
import torchvision.transforms as T
from PIL import ImageOps
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score, confusion_matrix, ConfusionMatrixDisplay
from detector import load_image, ClipEncoder, ResNetEncoder, occlusion_heatmap, heat_overlay, nearest_neighbors

DATA_DIR = '/content/drive/MyDrive/ai_vs_real'          # your A1 dataset on Drive
OUT = '/content/drive/MyDrive/ai_vs_real_A2_results'    # tables and figures for the report
os.makedirs(OUT, exist_ok=True)
SEED = 42
np.random.seed(SEED); torch.manual_seed(SEED)

## 1. Data and split (70 / 15 / 15)

In [ ]:
class_names = sorted(d for d in os.listdir(DATA_DIR) if os.path.isdir(os.path.join(DATA_DIR, d)))
AI_IDX = [i for i, c in enumerate(class_names) if 'ai' in c.lower()][0]
paths, labels = [], []
for i, c in enumerate(class_names):
    for f in sorted(os.listdir(f'{DATA_DIR}/{c}')):
        if f.lower().endswith(('.jpg', '.jpeg', '.png', '.webp')):
            paths.append(f'{DATA_DIR}/{c}/{f}'); labels.append(i)
paths, labels = np.array(paths), np.array(labels)
print(class_names, [int((labels == i).sum()) for i in range(2)])

p_train, p_tmp, y_train, y_tmp = train_test_split(paths, labels, test_size=0.30, stratify=labels, random_state=SEED)
p_val, p_test, y_val, y_test = train_test_split(p_tmp, y_tmp, test_size=0.50, stratify=y_tmp, random_state=SEED)
print('train', len(p_train), 'val', len(p_val), 'test', len(p_test))

## 2. Feature extraction

In [ ]:
def feats(enc, plist, perturb=None, bs=32):
    out = []
    for i in range(0, len(plist), bs):
        imgs = [load_image(p) for p in plist[i:i + bs]]
        if perturb: imgs = [perturb(im) for im in imgs]
        out.append(enc.encode(imgs))
    return np.concatenate(out)

BASE, CLIP = 'ResNet50 (baseline)', 'CLIP ViT-L/14'
enc = {BASE: ResNetEncoder(), CLIP: ClipEncoder('ViT-L-14', 'openai')}
X = {n: {s: feats(enc[n], pl) for s, pl in [('train', p_train), ('val', p_val), ('test', p_test)]} for n in enc}

## 3. Baseline vs SOTA (same classifier, C chosen on validation set)

In [ ]:
def fit(Xtr, ytr, Xva, yva):
    best = max(((make_pipeline(StandardScaler(), LogisticRegression(C=C, max_iter=5000)).fit(Xtr, ytr)) for C in (1e-4, 1e-3, 1e-2, 1e-1, 1)),
               key=lambda m: m.score(Xva, yva))
    return best

def metrics(clf, Xte, yte):
    pr = clf.predict_proba(Xte); pred = pr.argmax(1)
    p, r, f, _ = precision_recall_fscore_support(yte, pred, average='macro', zero_division=0)
    return {'Accuracy': accuracy_score(yte, pred), 'Precision': p, 'Recall': r, 'F1': f, 'AUC': roc_auc_score(yte, pr[:, 1])}

clfs = {n: fit(X[n]['train'], y_train, X[n]['val'], y_val) for n in enc}
rows = {n: metrics(clfs[n], X[n]['test'], y_test) for n in enc}
rows['A1 Keras ResNet50 (from A1 report)'] = {'Accuracy': 0.9375, 'Precision': 0.94, 'Recall': 0.94, 'F1': 0.94, 'AUC': np.nan}
results = pd.DataFrame(rows).T.round(3)
results.to_csv(f'{OUT}/main_results.csv'); results

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 4))
for ax, n in zip(axes, enc):
    ConfusionMatrixDisplay(confusion_matrix(y_test, clfs[n].predict(X[n]['test'])), display_labels=class_names).plot(ax=ax, cmap='Blues', colorbar=False)
    ax.set_title(n)
plt.tight_layout(); plt.savefig(f'{OUT}/confusion_matrices.png', dpi=200); plt.show()

## 4. Ablation: training with and without augmentation

In [ ]:
aug = T.Compose([T.RandomHorizontalFlip(), T.RandomResizedCrop(224, scale=(0.5, 1.0)), T.ColorJitter(0.2, 0.2)])
rows = {}
for n in enc:
    Xa = np.concatenate([X[n]['train']] + [feats(enc[n], p_train, perturb=aug) for _ in range(3)])
    clf_a = fit(Xa, np.tile(y_train, 4), X[n]['val'], y_val)
    rows[n] = {'Acc no aug': accuracy_score(y_test, clfs[n].predict(X[n]['test'])),
               'Acc with aug': accuracy_score(y_test, clf_a.predict(X[n]['test']))}
ablation = pd.DataFrame(rows).T.round(3); ablation.to_csv(f'{OUT}/ablation.csv'); ablation

## 5. Explainability: occlusion heatmap and nearest neighbours (CLIP model)

In [ ]:
pred = clfs[CLIP].predict(X[CLIP]['test'])
sel = [i for i in range(len(y_test)) if y_test[i] == 0 and pred[i] == 0][:1] + \
      [i for i in range(len(y_test)) if y_test[i] == 1 and pred[i] == 1][:1] + \
      [i for i in range(len(y_test)) if pred[i] != y_test[i]][:2]

fig, axes = plt.subplots(2, len(sel), figsize=(3.2 * len(sel), 6.6), squeeze=False)
for j, i in enumerate(sel):
    heat, base = occlusion_heatmap(enc[CLIP], clfs[CLIP], load_image(p_test[i]), grid=7, target=int(pred[i]))
    axes[0, j].imshow(base); axes[0, j].set_title(f'true: {class_names[y_test[i]]}\npred: {class_names[pred[i]]}', fontsize=9)
    axes[1, j].imshow(heat_overlay(base, heat)); axes[1, j].set_title('heatmap (red = important)', fontsize=9)
    for a in axes[:, j]: a.axis('off')
plt.tight_layout(); plt.savefig(f'{OUT}/explain_heatmap.png', dpi=200); plt.show()

In [ ]:
fig, axes = plt.subplots(2, 6, figsize=(15, 5.4))
for r, i in enumerate(sel[:2]):
    idx, sims = nearest_neighbors(X[CLIP]['test'][i], X[CLIP]['train'], k=5)
    axes[r, 0].imshow(ImageOps.fit(load_image(p_test[i]), (224, 224))); axes[r, 0].set_title('QUERY', fontsize=9)
    for k, (j, s) in enumerate(zip(idx, sims)):
        axes[r, k + 1].imshow(ImageOps.fit(load_image(p_train[j]), (224, 224)))
        axes[r, k + 1].set_title(f'{class_names[y_train[j]]}  sim={s:.2f}', fontsize=9)
for a in axes.ravel(): a.axis('off')
plt.tight_layout(); plt.savefig(f'{OUT}/explain_neighbors.png', dpi=200); plt.show()

## 6. Latency and environment

In [ ]:
imgs = [load_image(p) for p in p_test[:20]]
env = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'
rows = []
for n in enc:
    ts = []
    for im in imgs:
        t0 = time.perf_counter(); clfs[n].predict_proba(enc[n].encode([im]))
        if torch.cuda.is_available(): torch.cuda.synchronize()
        ts.append((time.perf_counter() - t0) * 1000)
    rows.append({'Model': n, 'Device': env, 'Latency ms': round(np.mean(ts[3:]), 1)})
latency = pd.DataFrame(rows); latency.to_csv(f'{OUT}/latency.csv', index=False); latency

## 7. Export for the web app

In [ ]:
ART = '/content/artifacts'; os.makedirs(f'{ART}/gallery', exist_ok=True)
joblib.dump(dict(clf=clfs[CLIP], class_names=class_names, ai_idx=AI_IDX, arch='ViT-L-14', pretrained='openai'), f'{ART}/probe.joblib')
files = []
for i, p in enumerate(p_train):
    fn = f'{i:04d}.jpg'; ImageOps.fit(load_image(p), (224, 224)).save(f'{ART}/gallery/{fn}', quality=90); files.append(fn)
np.savez(f'{ART}/gallery.npz', emb=X[CLIP]['train'].astype(np.float32), labels=y_train, files=np.array(files))
shutil.make_archive('/content/artifacts', 'zip', ART)
from google.colab import files as colab_files
colab_files.download('/content/artifacts.zip')